# Janus-Pro Base Model Test (download + inference)
This notebook is for **testing the base model** (`deepseek-ai/Janus-Pro-1B`) before any fine-tuning.

What it does:
- Installs the required Python packages
- Downloads the model from Hugging Face (first run can take a while)
- Runs a simple multimodal prompt (image + rubric) and prints the model output

> Note: You’ll need an environment with enough compute to run the model (ideally a GPU).

In [1]:
# 1) Environment check
import sys, platform
import torch

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print('VRAM (GB):', round(vram_gb, 2))
else:
    print('CPU-only mode detected (no NVIDIA/CUDA).')
    print('This base-model test can still run, but it will be very slow on CPU for a 1B VLM.')
    print('Tip: for a fast sanity check, run this notebook on Colab GPU.')

Python: 3.12.12
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM (GB): 14.74


In [2]:
# 2) Install dependencies (from requirements.txt)
# This installs into the *current* notebook environment (Jupyter kernel).
import os, sys, subprocess, platform

# If requirements.txt is present, use it. Otherwise write a small fallback file so the notebook still runs.
req_path = 'requirements.txt'
if not os.path.exists(req_path):
    # NOTE: This fallback is OS-aware so Windows CPU installs don’t try Linux-only wheels.
    req_path = os.path.join(os.getcwd(), 'requirements_fallback.txt')

    base_lines = [
        # Pin an exact Transformers 4.x build to avoid Colab silently upgrading within 4.x.
        # Janus-Pro remote code has been most reliable with this version in practice.
        'transformers==4.46.3',
        'accelerate>=0.33',
        'huggingface_hub',
        'torchvision',
        'pillow',
        'timm>=0.9.16',
        'attrdict',
        'einops',
        'sentencepiece',
        'requests',
    ]

    # Optional accelerators: only include where they commonly work out-of-the-box
    if platform.system() == 'Linux' and platform.machine().lower() in ('x86_64', 'amd64'):
        base_lines += [
            'bitsandbytes>=0.43',
            'hf-transfer',
        ]

    with open(req_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(base_lines) + '\n')

    print('requirements.txt not found in this runtime; wrote fallback to:', req_path)
else:
    print('Using requirements file:', os.path.abspath(req_path))

def _pip(cmd):
    full = [sys.executable, '-m', 'pip'] + cmd
    print('Running:', ' '.join(full))
    subprocess.check_call(full)

def _pip_show_version(pkg: str) -> str:
    try:
        out = subprocess.check_output([sys.executable, '-m', 'pip', 'show', pkg], text=True)
        line = next((l for l in out.splitlines() if l.startswith('Version:')), '')
        return line.replace('Version:', '').strip()
    except Exception:
        return ''

# Upgrade pip
_pip(['install', '-U', 'pip'])

# IMPORTANT (Colab): if Transformers 5.x (or another 4.x) is already installed, uninstall first.
# Even if we reinstall, the *already-imported* module stays in memory until runtime restart.
_pip(['uninstall', '-y', 'transformers', 'tokenizers'])

# Install our pinned Transformers build first (then install the rest).
_pip(['install', '-U', 'transformers==4.46.3'])

# Now install everything else (requirements file includes the same pin too).
_pip(['install', '-U', '-r', req_path])

print('pip show transformers:', _pip_show_version('transformers'))
print('pip show tokenizers:', _pip_show_version('tokenizers'))
print('pip show accelerate:', _pip_show_version('accelerate'))

if 'transformers' in sys.modules:
    print('NOTE: transformers is already imported in this Python process.')
    print('You MUST restart the runtime/kernel now, then re-run from Cell 2.')
else:
    print('Transformers not imported yet in this kernel (good).')

print('Done. If you see any load issues, restart the runtime/kernel and re-run from Cell 2.')

requirements.txt not found in this runtime; wrote fallback to: /content/requirements_fallback.txt
Running: /usr/bin/python3 -m pip install -U pip
Running: /usr/bin/python3 -m pip uninstall -y transformers tokenizers
Running: /usr/bin/python3 -m pip install -U transformers==4.46.3
Running: /usr/bin/python3 -m pip install -U -r /content/requirements_fallback.txt
pip show transformers: 4.46.3
pip show tokenizers: 0.20.3
pip show accelerate: 1.12.0
Transformers not imported yet in this kernel (good).
Done. If you see any load issues, restart the runtime/kernel and re-run from Cell 2.


In [8]:
# 2c) Install DeepSeek Janus code (REQUIRED for Janus-Pro weights)
# The model repo does NOT ship remote code, so we install the official Janus package from GitHub.
# We use --no-deps to avoid pip trying to change Torch/Transformers versions underneath us.
import sys, subprocess, platform
INSTALL_JANUS_FROM_GITHUB = True  # keep True on Colab/Linux; set False if you can't install from git
def _pip(cmd):
    full = [sys.executable, '-m', 'pip'] + cmd
    print('Running:', ' '.join(full))
    subprocess.check_call(full)
if not INSTALL_JANUS_FROM_GITHUB:
    print('INSTALL_JANUS_FROM_GITHUB=False (skipping).')
    print('If the model load fails with multi_modality, set this to True, run, then restart runtime.')
else:
    # On Windows, this requires git to be installed in the environment. Colab is OK.
    if platform.system() == 'Windows':
        print('Windows detected: this step requires git to be installed and on PATH.')
    _pip(['install', '-U', '--no-deps', "janus @ git+https://github.com/deepseek-ai/Janus.git"])
    print('Installed Janus code package.')
    print('IMPORTANT: If Transformers was already imported earlier, restart the runtime/kernel now.')

Running: /usr/bin/python3 -m pip install -U --no-deps janus @ git+https://github.com/deepseek-ai/Janus.git
Installed Janus code package.
IMPORTANT: If Transformers was already imported earlier, restart the runtime/kernel now.


In [3]:
# 2b) (Optional) If Janus load still fails: re-pin Transformers + restart reminder
import sys, subprocess

# Set this to True only if the model load in Cell 6 still errors (multi_modality / trust_remote_code / dynamic module issues).
INSTALL_COMPAT_TRANSFORMERS = False

def _pip(cmd):
    full = [sys.executable, '-m', 'pip'] + cmd
    print('Running:', ' '.join(full))
    subprocess.check_call(full)

if not INSTALL_COMPAT_TRANSFORMERS:
    print('INSTALL_COMPAT_TRANSFORMERS=False (skipping).')
    print('If Cell 6 still errors, set this to True, run this cell, then RESTART the runtime/kernel.')
else:
    _pip(['uninstall', '-y', 'transformers', 'tokenizers'])
    _pip(['install', '-U', 'transformers==4.46.3', 'accelerate>=0.33'])
    print('Installed Transformers 4.46.3. Restart the runtime/kernel, then run from Cell 3 onward.')

INSTALL_COMPAT_TRANSFORMERS=False (skipping).
If Cell 6 still errors, set this to True, run this cell, then RESTART the runtime/kernel.


In [4]:
# 3) Hugging Face login (optional)
import os
from huggingface_hub import login

# This cell tries, in order:
# 1) Environment variable HF_TOKEN
# 2) A local .env file in the current working directory

def _load_hf_token_from_dotenv(dotenv_path: str = '.env') -> str:
    if not os.path.exists(dotenv_path):
        return ''
    token = ''
    with open(dotenv_path, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if '=' not in line:
                continue
            key, val = line.split('=', 1)
            key = key.strip()
            val = val.strip().strip('"').strip("'")
            if key == 'HF_TOKEN':
                token = val
                break
    return token.strip()

HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    HF_TOKEN = _load_hf_token_from_dotenv('.env')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged in to Hugging Face (token loaded).')
else:
    print('HF_TOKEN not found in environment or .env. If the model is public for you, you can continue without login.')

HF_TOKEN not found in environment or .env. If the model is public for you, you can continue without login.


In [9]:
# 4) Load Janus-Pro base model + processor
import torch
import transformers
from transformers import AutoModelForCausalLM

MODEL_ID = 'deepseek-ai/Janus-Pro-1B'
print('Transformers version:', transformers.__version__)

# Choose device/dtype
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32
print('Device:', device, '| dtype:', dtype)

# Optional: try 4-bit if on GPU (saves VRAM). CPU mode never uses 4-bit.
LOAD_IN_4BIT = (device == 'cuda')
bnb_cfg = None
if LOAD_IN_4BIT:
    try:
        from transformers import BitsAndBytesConfig
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
        print('4-bit enabled.')
    except Exception as e:
        print('bitsandbytes not available; continuing without 4-bit. Error:', e)
        bnb_cfg = None

# Janus-Pro requires the Janus code package to be imported so it can register
# the `multi_modality` config/model into Transformers' AutoConfig/AutoModel mappings.
try:
    import janus  # noqa: F401
    from janus.models import VLChatProcessor  # noqa: F401
    # Importing modeling_vlm triggers AutoConfig.register(...) and AutoModelForCausalLM.register(...).
    import janus.models.modeling_vlm  # noqa: F401
    print('Janus package import OK (multi_modality registered).')
except Exception as e:
    raise RuntimeError(
        'Janus code package is not available, so Transformers cannot load model_type="multi_modality".\n'
        'Run Cell 3 (install Janus from GitHub), restart runtime, then re-run from Cell 2 onward.\n'
        f'Import error: {e}'
    )

def _build_model_kwargs():
    # NOTE: trust_remote_code is NOT needed here because code comes from the installed `janus` package, not HF repo.
    kwargs = dict(trust_remote_code=False)
    if device == 'cuda':
        kwargs.update(dict(
            device_map='auto',
            torch_dtype=dtype,
        ))
        if bnb_cfg is not None:
            kwargs['quantization_config'] = bnb_cfg
    else:
        kwargs.update(dict(low_cpu_mem_usage=True))
    return kwargs

model_kwargs = _build_model_kwargs()

# Load model
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
except Exception as e:
    # Common failure: bitsandbytes/4-bit incompatibility for this composite model; retry without quantization.
    if device == 'cuda' and bnb_cfg is not None:
        print('Model load failed with 4-bit; retrying without quantization. Error:', e)
        model_kwargs_noq = dict(model_kwargs)
        model_kwargs_noq.pop('quantization_config', None)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs_noq)
    else:
        raise

if device == 'cpu':
    model = model.to('cpu')
model.eval()

print('Loaded model type:', type(model))
print('Has language_model:', hasattr(model, 'language_model'))
print('Has prepare_inputs_embeds:', hasattr(model, 'prepare_inputs_embeds'))

# Load processor/tokenizer
from janus.models import VLChatProcessor
processor = VLChatProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
print('Loaded processor/tokenizer. EOS:', tokenizer.eos_token_id)

Transformers version: 4.46.3
Device: cuda | dtype: torch.bfloat16
4-bit enabled.
Python version is above 3.10, patching the collections module.


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:520: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


Janus package import OK (multi_modality registered).


pytorch_model.bin:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

Loaded model type: <class 'janus.models.modeling_vlm.MultiModalityCausalLM'>
Has language_model: True
Has prepare_inputs_embeds: True


...

special_tokens_map.json:   0%|          | 0.00/344 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


processor_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Some kwargs in processor config are unused and will not have any effect: image_tag, mask_prompt, num_image_tokens, ignore_id, add_special_token, sft_format. 


Loaded processor/tokenizer. EOS: 100001


In [14]:
# 5) Run a base-model inference test (image + rubric)
import json
import requests
from PIL import Image
from io import BytesIO
import torch

# Option A: provide a local image path (recommended)
IMAGE_PATH = '/content/Image2.jpg'  # e.g. r'C:\\path\\to\\image.png'

# Option B: if IMAGE_PATH is empty, download a small public sample image
SAMPLE_URL = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png'

if IMAGE_PATH.strip():
    img = Image.open(IMAGE_PATH).convert('RGB')
    print('Loaded image:', IMAGE_PATH, '| size:', img.size)
else:
    r = requests.get(SAMPLE_URL, timeout=30)
    r.raise_for_status()
    img = Image.open(BytesIO(r.content)).convert('RGB')
    print('Downloaded sample image:', SAMPLE_URL, '| size:', img.size)

# Example prompt + rubric (replace these with your own)
prompt = 'Prediction AccuracyPerfect. The calculation for $\hat{y} = m_1x_1 + m_2x_2 + b$ is correctly executed, yielding $9$.Error CalculationCorrect. The residual $\epsilon = y - \hat{y} = 10 - 9 = 1$ is properly identified.Gradient DerivationStrong. The formulas for the gradients of the cost function relative to weights ($m_1, m_2$) and bias ($b$) are applied correctly based on the Mean Squared Error derivative: $\frac{\partial J}{\partial w} = -\epsilon \cdot x$.Notation & ClarityGood. Symbols like partial derivatives ($\partial$) are used correctly. Minor point: Ensuring the cost function $J$ is explicitly defined (e.g., as $1/2 \epsilon^2$) would make the derivation 100% formal.'
rubric = 'Prediction AccuracyPerfect. The calculation for $\hat{y} = m_1x_1 + m_2x_2 + b$ is correctly executed, yielding $9$.Error CalculationCorrect. The residual $\epsilon = y - \hat{y} = 10 - 9 = 1$ is properly identified.Gradient DerivationStrong. The formulas for the gradients of the cost function relative to weights ($m_1, m_2$) and bias ($b$) are applied correctly based on the Mean Squared Error derivative: $\frac{\partial J}{\partial w} = -\epsilon \cdot x$.Notation & ClarityGood. Symbols like partial derivatives ($\partial$) are used correctly. Minor point: Ensuring the cost function $J$ is explicitly defined (e.g., as $1/2 \epsilon^2$) would make the derivation 100% formal.'

system = (
    'Grade this as a teacher  . '
    'Return STRICT JSON only (no markdown).'
)

# IMPORTANT: Janus expects <image_placeholder> in the user content
user = (
    '<image_placeholder>\n'
    f'Assignment:\n{prompt}\n\nRubric:\n{rubric}\n\n'
    'Student work is provided as image(s).\n\n'
    'Task: Produce a JSON grade with per-criterion scores, evidence, and feedback.'
)

conversation = [
    {
        'role': 'User',
        'content': system + '\n\n' + user,
        'images': [img],
    },
    {'role': 'Assistant', 'content': ''},
 ]

prepare_inputs = processor(conversations=conversation, images=[img], force_batchify=True)
if hasattr(prepare_inputs, 'to'):
    # On CPU we avoid dtype-casting; keep default tensors.
    if device == 'cuda':
        prepare_inputs = prepare_inputs.to(device, dtype=dtype)
    else:
        prepare_inputs = prepare_inputs.to(device)

if not hasattr(model, 'prepare_inputs_embeds') or not hasattr(model, 'language_model'):
    raise RuntimeError('This model object does not look like Janus-Pro (missing prepare_inputs_embeds/language_model).')

inputs_embeds = model.prepare_inputs_embeds(**prepare_inputs)

# CPU-only generation is very slow; keep the output short.
max_new = 64 if device == 'cpu' else 256
print('Generating with max_new_tokens =', max_new)

gen_kwargs = dict(
    inputs_embeds=inputs_embeds,
    attention_mask=prepare_inputs.attention_mask,
    max_new_tokens=max_new,
    do_sample=False,
    use_cache=True,
    pad_token_id=tokenizer.eos_token_id,
    bos_token_id=getattr(tokenizer, 'bos_token_id', None) or tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

with torch.no_grad():
    out = model.language_model.generate(**gen_kwargs)

text = tokenizer.decode(out[0].tolist(), skip_special_tokens=True)
print('\n=== Raw model output ===')
print(text)

# Optional: try to extract/parse JSON from the output
print('\n=== JSON parse attempt ===')
try:
    start = text.find('{')
    end = text.rfind('}')
    candidate = text[start:end+1] if start != -1 and end != -1 and end > start else text
    obj = json.loads(candidate)
    print(json.dumps(obj, indent=2))
except Exception as e:
    print('Could not parse JSON. Error:', e)

<>:24: SyntaxWarning: invalid escape sequence '\h'
<>:25: SyntaxWarning: invalid escape sequence '\h'
<>:24: SyntaxWarning: invalid escape sequence '\h'
<>:25: SyntaxWarning: invalid escape sequence '\h'
/tmp/ipython-input-671965783.py:24: SyntaxWarning: invalid escape sequence '\h'
  prompt = 'Prediction AccuracyPerfect. The calculation for $\hat{y} = m_1x_1 + m_2x_2 + b$ is correctly executed, yielding $9$.Error CalculationCorrect. The residual $\epsilon = y - \hat{y} = 10 - 9 = 1$ is properly identified.Gradient DerivationStrong. The formulas for the gradients of the cost function relative to weights ($m_1, m_2$) and bias ($b$) are applied correctly based on the Mean Squared Error derivative: $\frac{\partial J}{\partial w} = -\epsilon \cdot x$.Notation & ClarityGood. Symbols like partial derivatives ($\partial$) are used correctly. Minor point: Ensuring the cost function $J$ is explicitly defined (e.g., as $1/2 \epsilon^2$) would make the derivation 100% formal.'
/tmp/ipython-input-

Loaded image: /content/Image2.jpg | size: (810, 1080)
Generating with max_new_tokens = 256

=== Raw model output ===


Grade:

- Prediction AccuracyPerfect: 90
- Prediction AccuracyCorrect: 90
- Prediction AccuracyStrong: 90
- Prediction AccuracyGood: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Prediction AccuracyMinor Error: 90
- Pre

In [16]:
# 6) Sanity-check: describe what's in the image (no grading)
import torch
from PIL import Image
from io import BytesIO

# Reuse the image from the previous cell if it exists; otherwise load/generate one.
if 'img' in globals() and isinstance(img, Image.Image):
    _img = img
    print('Reusing img from previous cell | size:', _img.size)
else:
    # Fall back to IMAGE_PATH if set; else generate a simple sample plot image
    IMAGE_PATH = globals().get('IMAGE_PATH', '')
    if isinstance(IMAGE_PATH, str) and IMAGE_PATH.strip():
        _img = Image.open(IMAGE_PATH).convert('RGB')
        print('Loaded image from IMAGE_PATH | size:', _img.size)
    else:
        # Minimal built-in sample so this cell can run standalone
        import matplotlib.pyplot as plt
        import numpy as np

        xs = np.linspace(-2, 2, 200)
        ys = 2 * xs + 1
        fig = plt.figure(figsize=(4, 4), dpi=160)
        ax = fig.add_subplot(1, 1, 1)
        ax.plot(xs, ys, label='y = 2x + 1')
        ax.axhline(0, color='black', linewidth=0.8)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(-2, 2)
        ax.set_ylim(-3, 5)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title('Sample plot')
        ax.grid(True, alpha=0.3)
        fig.tight_layout()

        buf = BytesIO()
        fig.savefig(buf, format='png')
        plt.close(fig)
        buf.seek(0)
        _img = Image.open(buf).convert('RGB')
        print('Generated a sample plot image | size:', _img.size)

if 'processor' not in globals() or 'model' not in globals() or 'tokenizer' not in globals():
    raise RuntimeError('Run Cell 7 (load model + processor) first so processor/model/tokenizer exist.')

system = (
    'You are a precise visual describer. '
    'Describe what you see in the image in 3-8 short bullet points. '
    'If the image contains text, quote the most important text exactly. '
    'Do not mention grading or rubrics.'
)
user = (
    '<image_placeholder>\n'
    'Describe the image content.'
)

conversation = [
    {
        'role': 'User',
        'content': system + '\n\n' + user,
        'images': [_img],
    },
    {'role': 'Assistant', 'content': ''},
 ]

prepare_inputs = processor(conversations=conversation, images=[_img], force_batchify=True)
if hasattr(prepare_inputs, 'to'):
    if device == 'cuda':
        prepare_inputs = prepare_inputs.to(device, dtype=dtype)
    else:
        prepare_inputs = prepare_inputs.to(device)

inputs_embeds = model.prepare_inputs_embeds(**prepare_inputs)

max_new = 128 if device == 'cpu' else 256
print('Generating description with max_new_tokens =', max_new)

with torch.no_grad():
    out = model.language_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=prepare_inputs.attention_mask,
        max_new_tokens=max_new,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=getattr(tokenizer, 'bos_token_id', None) or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

desc = tokenizer.decode(out[0].tolist(), skip_special_tokens=True)
print('\n=== Image description ===')
print(desc)

Reusing img from previous cell | size: (810, 1080)
Generating description with max_new_tokens = 256

=== Image description ===
The image is a handwritten note in a grid-lined notebook. It contains a list of updates and parameters, with a focus on visual analysis. The list is as follows:

1. "Update parameters"
2. "m_1 = m_2 = 0.2"
3. "y = 0.1X(x - 2) + 0.2"
4. "m_2 = m_1 = 0.2"
5. "2 = 0.1(x - 3) = 2.1"
6. "b = b = 0.2x"
7. "6 = 6.0x"
8. "60 kte = match 15"
9. "m_1 = 1/2, m_2 = 2/3, b = 1.7"

The note is written in a clear and legible handwriting, with the date "1/1/1" at the top right corner. The content appears to be a detailed breakdown of a mathematical or visual analysis task, possibly related to a mathematical problem or a visual problem.
